In [ ]:
# Cell 1: run OCR once (~13s/image), cache results, plot confidence distribution.
import base64
import html
import io
import os
import sqlite3
from pathlib import Path

from IPython.display import HTML, display
from PIL import Image

os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())

from src.automate.ocr import ocr_text, parse_totals
from src.constants import CENT, MIN_CONFIDENCE
from src.db.db import get_paths

N = 30
BIN = 0.01

rows = (
    sqlite3.connect(get_paths()[0])
    .execute(
        "SELECT image_path, total_net, total_vat, total_gross FROM invoices ORDER BY id LIMIT ?",
        (N,),
    )
    .fetchall()
)

results = []  # (image, truth, ocr_totals_or_None, confidence, raw_text)
for img, *truth in rows:
    text, conf = ocr_text(img)
    try:
        # MIN_CONFIDENCE gate bypassed here: cell 2 applies your own threshold.
        o = parse_totals(text, 1.0)
        got = [o.total_net, o.total_vat, o.total_gross]
    except ValueError:
        got = None
    results.append((img, truth, got, conf, text))

confs = [r[3] for r in results]
bins = {}
for c in confs:
    b = int(c / BIN) * BIN
    bins[b] = bins.get(b, 0) + 1
peak = max(bins.values())
bars = "".join(
    f"<tr><td>{b:.2f}-{b + BIN:.2f}</td>"
    f"<td><div style='background:#4a90d9;height:14px;width:{400 * n / peak}px'></div></td>"
    f"<td>{n}</td></tr>"
    for b, n in sorted(bins.items())
)
display(
    HTML(
        f"<h3>Mean OCR confidence per invoice (n={len(confs)})</h3>"
        f"<p>min {min(confs):.3f} | median {sorted(confs)[len(confs) // 2]:.3f} | "
        f"max {max(confs):.3f} | current gate MIN_CONFIDENCE = {MIN_CONFIDENCE}</p>"
        f"<table>{bars}</table>"
    )
)

In [ ]:
# Cell 2: pick a threshold; re-run this cell only (no OCR re-run).
THRESHOLD = 0.95
SHOW = "below"  # "below" = would escalate to Tier 2, "above" = accepted by Tier 1


def correct(truth, got):
    return got is not None and all(abs(a - b) <= CENT for a, b in zip(truth, got))


above = [r for r in results if r[3] >= THRESHOLD]
below = [r for r in results if r[3] < THRESHOLD]
summary = (
    f"<h3>THRESHOLD = {THRESHOLD}</h3>"
    f"<p>accepted by Tier 1: {len(above)}/{len(results)} "
    f"({sum(correct(r[1], r[2]) for r in above)} correct)<br>"
    f"escalated to Tier 2: {len(below)}/{len(results)} "
    f"({sum(correct(r[1], r[2]) for r in below)} were actually correct = wasted LLM calls)</p>"
)
display(HTML(summary))


def img_tag(path, width=550):
    im = Image.open(path)
    im.thumbnail((width, width * 2))
    buf = io.BytesIO()
    im.save(buf, "PNG")
    return f'<img src="data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}">'


for img, truth, got, conf, text in sorted(above if SHOW == "above" else below, key=lambda r: r[3]):
    cells = "".join(
        f"<tr><td>{name}</td><td>{t:,.2f}</td>"
        f"<td style='background:{'#d4edda' if g is not None and abs(g - t) <= CENT else '#f8d7da'}'>"
        f"{'-' if g is None else f'{g:,.2f}'}</td></tr>"
        for name, t, g in zip(["net", "VAT", "gross"], truth, got or [None] * 3)
    )
    display(
        HTML(
            f"<div style='display:flex;gap:24px;border-top:2px solid #888;padding:12px 0'>"
            f"<div>{img_tag(img)}</div><div style='min-width:360px'>"
            f"<h4>{Path(img).name} | confidence {conf:.3f}</h4>"
            f"<table><tr><th></th><th>ground truth</th><th>OCR</th></tr>{cells}</table>"
            f"<details><summary>raw OCR text</summary><pre>{html.escape(text)}</pre></details>"
            f"</div></div>"
        )
    )